# 04 · Dữ liệu train LoRA

Từ các lượt chạy **trên train** (có đáp án) dựng các dòng SFT cho LoRA, bằng rejection sampling:

1. **Lượt chạy có nhãn:** `02_inference` với `SPLIT = "train"`, `REUSE_LAYERS = False`, `SAVE_PROMPTS = True` (ví dụ `IDS = "splits/rft_train24.txt"`,
   2368 câu của 24 shop ngoài holdout). Mỗi câu có 8 mẫu thường và tới 4 mẫu thinking, kèm prompt đã gửi.
2. **Dòng SFT** (`training/build_sft.py`): mỗi câu giữ lời giải đúng trên route tốt nhất (macro công nợ → view chuẩn → bảng thô), câu khó
   (0-5/8 mẫu đúng) cho tới 2 dòng, câu dễ góp `EASY` dòng, mẫu thinking đúng thành dòng thinking. Lượt teacher (model lớn hơn chạy trên
   các câu 9B chưa giải được, chỉ trên train) đưa vào bằng `TEACHER_RUNS`.
3. **Ghi chú** (`training/add_notes.py`, cần GPU): dòng thường chưa có ghi chú được thêm vài dòng "Notes:" trước SQL, đúng định dạng lúc suy luận.
4. **Ngân hàng few-shot** (`training/build_fewshot_bank.py`), tùy chọn: mỗi câu một lời giải ngắn chỉ đọc view chuẩn.

Chỉ dùng dữ liệu train. Câu public / private không bao giờ vào dữ liệu train.

In [ ]:
# Setup. Local checkout: nothing to install. Kaggle: Internet on, GPU accelerator for inference / training, the competition data
# attached as a private dataset, and the code either attached as a dataset (a copy of the checkout) or cloned with your
# GitHub token in Add-ons -> Secrets as GITHUB_TOKEN (if the repo is private). Colab: Google Drive holding MyDrive/datamate/code and
# MyDrive/datamate/data (or a GITHUB_TOKEN secret for the code), a GPU runtime for inference / training.
import os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = "github.com/hana200804/datamate-agent-Locked-In1.git"


def clone(root, token):
    subprocess.run(["git", "clone", "--depth", "1", f"https://x-access-token:{token}@{REPO_URL}", str(root)], check=True)
    subprocess.run(["git", "remote", "set-url", "origin", f"https://{REPO_URL}"], cwd=root, check=True)  # token out of saved files


if Path("/kaggle/input").exists():
    root = Path("/tmp/datamate-agent-Locked-In1")
    attached = sorted(p.parent.parent for p in Path("/kaggle/input").glob("**/datamate/notebook.py"))
    if not root.exists() and attached:
        shutil.copytree(attached[0], root)                         # code attached as a dataset: no token needed
    if not root.exists():
        from kaggle_secrets import UserSecretsClient
        clone(root, UserSecretsClient().get_secret("GITHUB_TOKEN"))
elif "COLAB_RELEASE_TAG" in os.environ:
    from google.colab import drive
    drive.mount("/content/drive")
    root, code = Path("/content/datamate-agent-Locked-In1"), Path("/content/drive/MyDrive/datamate/code")
    if not root.exists() and (code / "datamate/notebook.py").exists():
        shutil.copytree(code, root)                                # code uploaded to Drive: no token needed
    if not root.exists():
        from google.colab import userdata                          # key icon -> secret GITHUB_TOKEN with notebook access
        clone(root, userdata.get("GITHUB_TOKEN"))
else:
    root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(root)
sys.path.insert(0, str(root))
from datamate.notebook import install, setup
install("llm")
env = setup()

In [ ]:
import json
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

read = lambda p: [json.loads(l) for l in open(p, encoding="utf-8-sig") if l.strip()]
pd.set_option("display.max_colwidth", 80)

In [ ]:
RUNS = ["runs/sql_v8_pool"]        # labeled train runs with --save-prompts (dirs holding trace.jsonl or shard*/trace.jsonl)
TEACHER_RUNS = []                  # optional teacher runs on train questions
OUT = "runs/sft/sql_rft_nb.jsonl"
EASY = 800                         # rows from easy questions (6-8 of 8 samples right)
MAX = 4000                         # cap on rows (debt and teacher rows are kept)
ADD_NOTES = True                   # needs a GPU (vLLM, base weights)
BUILD_BANK = False                 # also write a few-shot bank from RUNS

In [ ]:
def trace_dir(r):  # build_sft.py reads RUN/trace.jsonl: join the shards of a multi-GPU run first
    r = Path(r) if Path(r).is_absolute() else env.root / r
    if not (r / "trace.jsonl").exists() and list(r.glob("shard*/trace.jsonl")):
        with (r / "trace.jsonl").open("w", encoding="utf-8") as f:
            for p in sorted(r.glob("shard*/trace.jsonl")):
                f.write(p.read_text(encoding="utf-8"))
    return r
runs, teachers = [trace_dir(r) for r in RUNS], [trace_dir(r) for r in TEACHER_RUNS]
out = Path(OUT) if Path(OUT).is_absolute() else env.root / OUT
out.parent.mkdir(parents=True, exist_ok=True)
env.run([env.python, "training/build_sft.py", *runs, *(["--teacher", *teachers] if teachers else []), "--fewshot", "assets/fewshot_bank.jsonl",
         "--out", out, "--max", MAX, "--easy", EASY], log=out.with_suffix(".log"), show="available|rows by|Traceback|Error")

In [ ]:
sft = out
if ADD_NOTES and not env.can_run_llm:                   # the notes step needs the model: take the notes file saved from a 24 GB GPU run
    sft = env.use_saved(out.stem + "_notes.jsonl", out.with_name(out.stem + "_notes.jsonl"))
elif ADD_NOTES:
    flags, _ = env.inference_flags()                     # checks the GPU, Ampere runs bf16 (no fp8)
    sft = out.with_name(out.stem + "_notes.jsonl")
    env.run([env.python, "training/add_notes.py", out, sft, "--model", env.base_model(), "--rulebook", env.data / "train/rulebook.md",
             "--quant", "fp8" if "--quant" in flags else "none"], log=sft.with_suffix(".log"), show="notes on|Traceback|Error")
print("SFT file for 05_train_lora:", sft)

## Dữ liệu SFT trông thế nào

Số dòng theo route, tỷ lệ dòng thinking, độ dài (prompt chiếm phần lớn mỗi dòng, loss chỉ tính trên phần trả lời).

In [ ]:
rows = read(sft)
df = pd.DataFrame([{"id": r["id"], "route": r.get("route"), "thinking": bool(r.get("thinking")), "prompt_chars": len(r["system"]) + len(r["user"]),
                    "reply_chars": len(r["reply"]), "notes": r["reply"].startswith("Notes:")} for r in rows])
print(len(df), "rows,", df["id"].nunique(), "questions")
display(df.groupby(["route", "thinking"]).size().unstack(fill_value=0))
display(df[["prompt_chars", "reply_chars"]].describe().round(0))
print(f"plain rows with notes: {df[~df.thinking].notes.mean():.1%}")

In [ ]:
r = rows[0]
print(r["user"][-600:], "\n---- reply ----\n", r["reply"][:800])

In [ ]:
if BUILD_BANK:
    env.run([env.python, "training/build_fewshot_bank.py", *runs, "--out", out.with_name("fewshot_bank.jsonl")], show="Traceback|Error|bank|rows")